## CARGA ENVIROMENT SPARK - CONEXIONES A SERVIDOR LOCAL + MINIO

In [ ]:
%%capture
import os
from pathlib import Path
from dotenv import load_dotenv
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--driver-memory 2g "
    "--packages org.apache.hadoop:hadoop-aws:3.5.0 "
    "pyspark-shell"
)
load_dotenv(Path.cwd() / "minio.env", override=True)
for nombre in ("MINIO_ENDPOINT", "MINIO_ACCESS_KEY", "MINIO_SECRET_KEY"):
    if not os.getenv(nombre):
        raise ValueError(f"Falta {nombre} en minio.env")

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("Leer egresos")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.files.maxPartitionBytes", "32m")
    .config("spark.hadoop.fs.s3a.input.stream.type", "classic")
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["MINIO_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.access.key", os.environ["MINIO_ACCESS_KEY"])
    .config("spark.hadoop.fs.s3a.secret.key", os.environ["MINIO_SECRET_KEY"])
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .getOrCreate()
)
spark.conf.set("spark.sql.legacy.javaCharsets", "true")
spark.sparkContext.setLogLevel("ERROR")
spark._jvm.java.lang.Thread.currentThread().getContextClassLoader().loadClass(
    "org.apache.hadoop.fs.s3a.S3AFileSystem"
)

## INFERIR LOS CODIGOS Y SUS GLOSAS ASOCIADAS ... AGRUPO PARA OBTENER LAS LLAVE GLOSA O CODIGO

In [ ]:
from pyspark.sql import functions as F
df = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("inferSchema", "false")
    .csv("s3a://doctorado/data/egresos/egresos_hospitalarios_2024.csv")
)
enfermedades = (
    df.select(
        F.regexp_extract(
            F.upper(F.trim(F.col("cau_cie10"))),
            r"^(\S+)", 1
        ).alias("codigo"),
        F.trim(
            F.regexp_replace(
                F.trim(F.col("cau_cie10")),
                r"^\S+\s*", ""
            )
        ).alias("nombre")
    )
    .filter(F.col("codigo").isin("G300", "G301", "G308", "G309"))
    .dropDuplicates(["codigo", "nombre"])
    .orderBy("codigo", "nombre")
)

df_enfermedades = enfermedades.select("codigo", "nombre")
pdf = enfermedades.toPandas()
display(pdf)


## AQUI GENERO FUNCION PARA LEER DESDE EL STORAGE + SPARK EL CSV POR ANIO DEPENDIENDO SI ES CODIGO O GLOSA

In [ ]:
from pyspark.sql import functions as F

def leer_egresos(
    anio,
    filtrar_por="codigo",
    codigos=None,
    catalogo=None,
    ruta_base="s3a://doctorado/data/egresos",
    encoding= "iso-8859-1"
):
    """Devuelve un DataFrame Spark filtrado por código o nombre."""
    if filtrar_por not in ("codigo", "nombre"):
        raise ValueError("filtrar_por debe ser 'codigo' o 'nombre'")

    if filtrar_por == "nombre" and catalogo is None:
        raise ValueError("Debes proporcionar catalogo para filtrar por nombre")

    if codigos is None:
        codigos = ["G300", "G301", "G308", "G309"]

    def normalizar(columna):
        return F.lower(
            F.trim(F.regexp_replace(columna, r"\s+", " "))
        )

    tabla = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("inferSchema", "false")
        .option("encoding", encoding)
        .csv(
            f"{ruta_base.rstrip('/')}/"
            f"egresos_hospitalarios_{anio}.csv"
        )
        .withColumn(
            "codigo",
            F.regexp_replace(
                F.regexp_extract(
                    F.upper(F.trim(F.col("cau_cie10"))),
                    r"^([A-Z][0-9]{2}\.?[0-9A-Z]*)(?=\s|$)",
                    1
                ),
                r"\.",
                ""
            )
        )
        .withColumn(
            "nombre",
            F.trim(
                F.regexp_replace(
                    F.trim(F.col("cau_cie10")),
                    r"(?i)^[A-Z][0-9]{2}\.?[0-9A-Z]*\s+",
                    ""
                )
            )
        )
    )

    if filtrar_por == "codigo":
        return tabla.filter(F.col("codigo").isin(codigos))

    nombres = (
        catalogo
        .select(normalizar(F.col("nombre")).alias("_nombre_filtro"))
        .distinct()
    )

    return (
        tabla
        .withColumn("_nombre_filtro", normalizar(F.col("nombre")))
        .join(F.broadcast(nombres), on="_nombre_filtro", how="left_semi")
        .drop("_nombre_filtro")
    )

## CONSOLIDADO TABLA

### prov_ubi
* 01 Azuay
* 02 Bolívar
* 03 Cañar
* 04 Carchi
* 05 Cotopaxi
* 06 Chimborazo
* 07 El Oro
* 08 Esmeraldas
* 09 Guayas
* 10 Imbabura
* 11 Loja
* 12 Los Ríos
* 13 Manabí
* 14 Morona Santiago
* 15 Napo
* 16 Pastaza
* 17 Pichincha
* 18 Tungurahua
* 19 Zamora Chinchipe
* 20 Galápagos
* 21 Sucumbíos
* 22 Orellana
* 23 Santo Domingo de los Tsáchilas
* 24 Santa Elena

### area_ubi
* 1 Urbano
* 2 Rural

### sector
* 1 Público
* 2 Privado con fines de lucro
* 3 Privado sin fines de lucro

### sexo
* 1 Hombre
* 2 Mujer

### etnia
* 1 Indígena
* 2 Afroecuatoriano/a Afrodescendiente
* 3 Negro/a
* 4 Mulato/a
* 5 Montubio/a
* 6 Mestizo/a
* 7 Blanco/a
* 8 Otro/a
* 9 Ignorado/a

### con_egrpa
* 1 Vivo
* 2 Fallecido menos de 48 horas
* 3 Fallecido en 48 horas y más

In [ ]:
from functools import reduce
from pyspark.sql import functions as F

diccionarios = {
    "prov_ubi": {
        1: "Azuay", 2: "Bolívar", 3: "Cañar", 4: "Carchi",
        5: "Cotopaxi", 6: "Chimborazo", 7: "El Oro",
        8: "Esmeraldas", 9: "Guayas", 10: "Imbabura",
        11: "Loja", 12: "Los Ríos", 13: "Manabí",
        14: "Morona Santiago", 15: "Napo", 16: "Pastaza",
        17: "Pichincha", 18: "Tungurahua",
        19: "Zamora Chinchipe", 20: "Galápagos",
        21: "Sucumbíos", 22: "Orellana",
        23: "Santo Domingo de los Tsáchilas", 24: "Santa Elena"
    },
    "area_ubi": {
        1: "Urbano", 2: "Rural"
    },
    "sector": {
        1: "Público",
        2: "Privado con fines de lucro",
        3: "Privado sin fines de lucro"
    },
    "sexo": {
        1: "Hombre", 2: "Mujer"
    },
    "etnia": {
        1: "Indígena",
        2: "Afroecuatoriano/a Afrodescendiente",
        3: "Negro/a",
        4: "Mulato/a",
        5: "Montubio/a",
        6: "Mestizo/a",
        7: "Blanco/a",
        8: "Otro/a",
        9: "Ignorado/a"
    },
    "con_egrpa": {
        1: "Vivo",
        2: "Fallecido menos de 48 horas",
        3: "Fallecido en 48 horas y más"
    }
}

enfermedades_por_codigo = {
    "G300": "Enfermedad de Alzheimer de comienzo temprano",
    "G301": "Enfermedad de Alzheimer de comienzo tardío",
    "G308": "Otros tipos de enfermedad de Alzheimer",
    "G309": "Enfermedad de Alzheimer, no especificada"
}

configuracion_por_anio = {
    2015: {"filtro": "nombre", "encoding": "windows-1252", "diccionario": False},
    2016: {"filtro": "codigo", "encoding": "windows-1252", "diccionario": True},
    2017: {"filtro": "nombre", "encoding": "windows-1252", "diccionario": False},
    2018: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": True},
    2019: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": True},
    2020: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": True},
    2021: {"filtro": "nombre", "encoding": "UTF-8", "diccionario": False},
    2022: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": False},
    2023: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": False},
    2024: {"filtro": "codigo", "encoding": "UTF-8", "diccionario": False},
}

correcciones_por_anio = {
    # Activa solo si sigue dañado después de leer con UTF-8:
    # 2024: {"sector": {"PÃºblico": "Público"}}
}

columnas = [
    "prov_ubi",
    "area_ubi",
    "sector",
    "sexo",
    "edad",
    "etnia",
    "anio_ingr",
    "dia_estad",
    "con_egrpa",
    "cau_cie10",
    "anio_ingesta",
]

spark.conf.set("spark.sql.legacy.javaCharsets", "true")


def crear_mapa(diccionario):
    return F.create_map(*[
        expresion
        for clave, valor in diccionario.items()
        for expresion in (F.lit(clave), F.lit(valor))
    ])


def normalizar_texto(columna):
    return F.lower(
        F.trim(F.regexp_replace(columna, r"\s+", " "))
    )


def extraer_codigo(columna):
    return F.regexp_replace(
        F.regexp_extract(
            F.upper(F.trim(columna)),
            r"^([A-Z][0-9]{2}\.?[0-9A-Z]*)(?=\s|$)",
            1
        ),
        r"\.",
        ""
    )


def extraer_nombre(columna):
    return F.trim(
        F.regexp_replace(
            F.trim(columna),
            r"(?i)^[A-Z][0-9]{2}\.?[0-9A-Z]*(?:\s+|$)",
            ""
        )
    )


# Catálogo independiente de las lecturas de otros años.
df_enfermedades = spark.createDataFrame(
    list(enfermedades_por_codigo.items()),
    ["codigo", "nombre"]
)


def leer_egresos(
    anio,
    filtrar_por="codigo",
    codigos=None,
    catalogo=None,
    ruta_base="s3a://doctorado/data/egresos",
    encoding="UTF-8"
):
    if filtrar_por not in ("codigo", "nombre"):
        raise ValueError("filtrar_por debe ser 'codigo' o 'nombre'")

    if codigos is None:
        codigos = list(enfermedades_por_codigo)

    tabla = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("inferSchema", "false")
        .option("encoding", encoding)
        .csv(
            f"{ruta_base.rstrip('/')}/"
            f"egresos_hospitalarios_{anio}.csv"
        )
        .withColumn("codigo", extraer_codigo(F.col("cau_cie10")))
        .withColumn("nombre", extraer_nombre(F.col("cau_cie10")))
    )

    if filtrar_por == "codigo":
        return tabla.filter(F.col("codigo").isin(codigos))

    if catalogo is None:
        raise ValueError("Debes proporcionar un catálogo para filtrar por nombre")

    nombres_filtro = (
        catalogo
        .select(normalizar_texto(F.col("nombre")).alias("_nombre_filtro"))
        .distinct()
    )

    return (
        tabla
        .withColumn("_nombre_filtro", normalizar_texto(F.col("nombre")))
        .join(
            F.broadcast(nombres_filtro),
            on="_nombre_filtro",
            how="left_semi"
        )
        .drop("_nombre_filtro")
    )


def convertir_categorias(tabla):
    for columna, categorias in diccionarios.items():
        if columna not in tabla.columns:
            continue

        valor = F.trim(F.col(columna).cast("string"))

        codigo = F.when(
            valor.rlike(r"^\d{1,2}(?:\.0+)?$"),
            valor.cast("double").cast("int")
        )

        tabla = tabla.withColumn(
            columna,
            F.coalesce(
                F.element_at(crear_mapa(categorias), codigo),
                F.col(columna).cast("string")
            )
        )

    return tabla


def normalizar_enfermedad(tabla):
    valor = F.col("cau_cie10")
    codigo = extraer_codigo(valor)
    nombre = extraer_nombre(valor)

    nombres_canonicos = {
        " ".join(nombre.split()).lower(): nombre
        for nombre in enfermedades_por_codigo.values()
    }

    return tabla.withColumn(
        "cau_cie10",
        F.coalesce(
            F.element_at(crear_mapa(enfermedades_por_codigo), codigo),
            F.element_at(
                crear_mapa(nombres_canonicos),
                normalizar_texto(nombre)
            ),
            F.trim(valor)
        )
    )


def cargar_anios(configuracion, correcciones=None):
    if not configuracion:
        raise ValueError("Debes configurar al menos un año")

    correcciones = correcciones or {}
    dataframes = {}

    for anio, opciones in configuracion.items():
        tabla = leer_egresos(
            anio,
            filtrar_por=opciones["filtro"],
            catalogo=df_enfermedades,
            encoding=opciones["encoding"]
        )

        if opciones["diccionario"]:
            tabla = convertir_categorias(tabla)

        for columna, reemplazos in correcciones.get(anio, {}).items():
            tabla = tabla.replace(reemplazos, subset=[columna])

        tabla = normalizar_enfermedad(tabla)

        if "area_ubi" in tabla.columns:
            tabla = tabla.withColumn(
                "area_ubi",
                F.when(
                    normalizar_texto(F.col("area_ubi")) == "urbana",
                    F.lit("Urbano")
                ).otherwise(F.col("area_ubi"))
            )

        # Año del archivo de origen.
        tabla = tabla.withColumn("anio_ingesta", F.lit(anio))

        dataframes[anio] = tabla

    unificado = reduce(
        lambda izquierda, derecha: izquierda.unionByName(
            derecha,
            allowMissingColumns=True
        ),
        dataframes.values()
    )

    return dataframes, unificado


dataframes, df_unificado = cargar_anios(
    configuracion_por_anio,
    correcciones_por_anio
)

df_final = df_unificado.select(*columnas)

pdf = df_final.limit(3000).toPandas()
display(pdf)

In [ ]:
pdf = df_final.toPandas().groupby(['anio_ingesta'])
display(pdf)

In [ ]:
pdf.count()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

datos = df_final.toPandas().copy()
categoricas = [
    "prov_ubi", "area_ubi", "sector",
    "sexo", "etnia", "con_egrpa", "cau_cie10"
]
numericas = ["edad", "dia_estad", "anio_ingr", "anio_ingesta"]


for columna in categoricas:
    datos[columna] = (
        datos[columna]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )


originales = datos[numericas].copy()

for columna in numericas:
    datos[columna] = pd.to_numeric(
        datos[columna].astype("string").str.strip(),
        errors="coerce"
    )

print(f"Registros: {len(datos):,}")
print(f"Columnas: {datos.shape[1]}")
print(f"Filas idénticas en las columnas seleccionadas: {datos.duplicated().sum():,}")


In [ ]:
calidad = pd.DataFrame({
    "tipo": datos.dtypes.astype(str),
    "faltantes": datos.isna().sum(),
    "faltantes_%": datos.isna().mean().mul(100).round(2),
    "valores_distintos": datos.nunique()
})
display(calidad)

conversiones = {}
for columna in numericas:
    texto = originales[columna].astype("string").str.strip()
    presente = texto.notna() & texto.ne("")
    conversiones[columna] = (
        presente & datos[columna].isna()
    ).sum()

display(
    pd.Series(conversiones, name="valores_no_numericos")
    .to_frame()
)

display(
    datos[["edad", "dia_estad"]]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95, 0.99])
    .round(2)
)

revision = pd.Series({
    "Edad negativa": (datos["edad"] < 0).sum(),
    "Edad mayor a 120 (revisar unidad/código)": (datos["edad"] > 120).sum(),
    "Estancia negativa": (datos["dia_estad"] < 0).sum(),
    "Estancia igual a cero": (datos["dia_estad"] == 0).sum(),
    "Filas con carácter dañado �": (
        datos[categoricas]
        .apply(lambda serie: serie.str.contains("�", regex=False, na=False))
        .any(axis=1)
        .sum()
    )
}, name="registros")

display(revision.to_frame())

In [ ]:
def frecuencias(columna):
    conteos = datos[columna].fillna("Sin dato").value_counts()
    return pd.DataFrame({
        "registros": conteos,
        "porcentaje": conteos.div(len(datos)).mul(100).round(2)
    })

for columna in categoricas:
    print(f"\n{columna}")
    display(frecuencias(columna))

resumen_anual = (
    datos.groupby("anio_ingesta")
    .agg(
        registros=("anio_ingesta", "size"),
        edad_mediana=("edad", "median"),
        estancia_mediana=("dia_estad", "median"),
        estancia_p95=("dia_estad", lambda serie: serie.quantile(0.95))
    )
    .round(2)
)

display(resumen_anual)

enfermedades_por_anio = pd.crosstab(
    datos["anio_ingesta"],
    datos["cau_cie10"]
)

display(enfermedades_por_anio)

display(
    pd.crosstab(
        datos["sexo"],
        datos["con_egrpa"],
        margins=True
    )
)

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(18, 10))

resumen_anual["registros"].plot(
    kind="bar", ax=ejes[0, 0], color="steelblue"
)
ejes[0, 0].set_title("Egresos seleccionados por año")
ejes[0, 0].set_xlabel("Año del archivo")
ejes[0, 0].set_ylabel("Registros")

datos["sexo"].fillna("Sin dato").value_counts().plot(
    kind="bar", ax=ejes[0, 1], color="teal"
)
ejes[0, 1].set_title("Distribución por sexo")
ejes[0, 1].tick_params(axis="x", rotation=0)

datos["edad"].dropna().plot(
    kind="hist", bins=20, ax=ejes[0, 2], color="slateblue"
)
ejes[0, 2].set_title("Edad registrada")
ejes[0, 2].set_xlabel("Edad: verificar unidad")
ejes[0, 2].set_ylabel("Registros")

datos["dia_estad"].dropna().plot(
    kind="hist", bins=30, ax=ejes[1, 0], color="darkorange"
)
ejes[1, 0].set_title("Días de estancia")
ejes[1, 0].set_xlabel("Días")
ejes[1, 0].set_ylabel("Registros")

datos["prov_ubi"].value_counts().head(10).sort_values().plot(
    kind="barh", ax=ejes[1, 1], color="seagreen"
)
ejes[1, 1].set_title("10 provincias con más registros")
ejes[1, 1].set_xlabel("Registros")

datos["con_egrpa"].fillna("Sin dato").value_counts().plot(
    kind="barh", ax=ejes[1, 2], color="indianred"
)
ejes[1, 2].set_title("Condición de egreso")
ejes[1, 2].set_xlabel("Registros")

plt.tight_layout()
plt.show()

ax = enfermedades_por_anio.plot(
    figsize=(12, 5),
    marker="o"
)
ax.set_title("Egresos por tipo de Alzheimer y año")
ax.set_xlabel("Año del archivo")
ax.set_ylabel("Registros")
ax.legend(title="Enfermedad", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
# Cantidad de filas duplicadas adicionales.
print(f"Duplicados adicionales: {datos.duplicated().sum():,}")

# Muestra todas las filas repetidas, incluida la primera aparición.
duplicados = datos.loc[datos.duplicated(keep=False)]
display(duplicados)

# Combinaciones repetidas y su frecuencia.
repeticiones = (
    datos.value_counts(dropna=False)
    .reset_index(name="repeticiones")
    .query("repeticiones > 1")
    .sort_values("repeticiones", ascending=False)
)

display(repeticiones)

In [ ]:
datos_sin_duplicados = datos.drop_duplicates()

print(f"Total sin duplicados: {len(datos_sin_duplicados):,}")

In [ ]:
# Nulos por columna.
display(datos.isna().sum().to_frame("nulos"))

# Total de celdas nulas.
print(f"Total de nulos: {datos.isna().sum().sum():,}")

# Filas con al menos un valor nulo.
filas_con_nulos = datos.loc[datos.isna().any(axis=1)]

print(f"Filas con nulos: {len(filas_con_nulos):,}")
display(filas_con_nulos)